# Fit reciprocal spectral organization for survival analysis
Fit the bounded two-term trajectory directly to $\mathcal O_\lambda=1/T_\lambda$ with no additive asymptote uncertainty floor. The historical trajectory column named `O_lambda` contains $T_\lambda$. The estimator takes the reciprocal once. The previous estimate table supplies only the cohort identities.

The run saves one patient summary and 16 equal-weight asymptote particles per patient. A completed output directory can be reopened by rerunning this notebook. To start a separate fit, select a new output directory.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import tempfile

import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
DATA = Path('/content/drive/MyDrive/Hartert/CAR-T-spectral-organization/DATA')
SOURCE = DATA / 'olambda_asymptotes_min1000'
TRAJECTORIES = SOURCE / 'combined_olambda_trajectory_points.csv'
COHORT = SOURCE / 'patient_olambda_asymptotes.csv'
OUTPUT = DATA / 'olambda_reciprocal_direct_no_asymptote_floor_v1'
for path in (TRAJECTORIES, COHORT):
    if not path.is_file():
        raise FileNotFoundError(path)

In [ ]:
# Use the validated revision in an isolated checkout, including on a reused runtime.
repo = Path(tempfile.mkdtemp(prefix='direct-olambda-')) / 'repo'
revision = '23d32724669281551430f7e651419b60f9a121f3'
subprocess.run(['git', 'clone', '--quiet', '--single-branch', '--branch', 'direct-olambda',
                'https://github.com/adamnik/CAR-T-spectral-organization.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--quiet', '--detach', revision], check=True)
if shutil.which('Rscript') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', 'r-base', 'r-cran-nlme'], check=True)
subprocess.run(['Rscript', '-e', 'if (!requireNamespace("nlme", quietly=TRUE)) install.packages("nlme", repos="https://cloud.r-project.org"); stopifnot(requireNamespace("nlme", quietly=TRUE))'], check=True)
print('Estimator revision:', revision)

In [ ]:
keys = ['Dataset', 'sample_id']
cohort = pd.read_csv(COHORT, usecols=keys)
assert len(cohort) == 163 and cohort.Dataset.nunique() == 8
assert not cohort.duplicated(keys).any() and cohort.notna().all().all()
points = pd.read_csv(TRAJECTORIES, usecols=keys + ['n_cells_available', 'target_depth', 'O_lambda'])
points = points.merge(cohort, on=keys, validate='many_to_one')
eligible = points.loc[(points.n_cells_available >= 1000) & np.isfinite(points.target_depth)
                      & np.isfinite(points.O_lambda) & (points.O_lambda > 1 + 1e-5)]
assert len(eligible) == 20992 and len(eligible[keys].drop_duplicates()) == len(cohort)
display(eligible.groupby('Dataset').agg(patients=('sample_id', 'nunique'), points=('target_depth', 'size')))

In [ ]:
est_path = OUTPUT / 'patient_olambda_asymptotes.csv'
particle_path = OUTPUT / 'patient_olambda_particles.csv'
if OUTPUT.exists():
    if not all(path.is_file() for path in (est_path, particle_path)):
        raise RuntimeError(f'Incomplete output at {OUTPUT}. Select a new OUTPUT directory.')
    print('Reading existing fit:', OUTPUT)
else:
    command = ['Rscript', repo / 'spectral_organization/estimation/fit_direct_olambda.R',
               TRAJECTORIES, COHORT, OUTPUT]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)

In [ ]:
est = pd.read_csv(est_path).set_index(keys, verify_integrity=True).sort_index()
pt = pd.read_csv(particle_path).set_index(keys, verify_integrity=True).sort_index()
expected = cohort.set_index(keys).sort_index().index
assert est.index.equals(expected) and pt.index.equals(expected)
assert est.olambda_scale.eq('reciprocal').all() and est.model.eq('bounded_direct_no_asymptote_floor').all()
assert list(pt.columns) == [f'particle_{k:02d}' for k in range(1, 17)]
a = pt.to_numpy()
assert ((a > 0) & (a < 1)).all() and (np.diff(a, axis=1) >= 0).all()
assert ((0 < est.O_lambda_inf_q025) & (est.O_lambda_inf_q025 < est.O_lambda_inf_central)
        & (est.O_lambda_inf_central < est.O_lambda_inf_q975) & (est.O_lambda_inf_q975 < 1)).all()
assert est.n_trajectory_points_available.sum() == len(eligible)
print(f'{len(est)} patients | {est.index.get_level_values("Dataset").nunique()} datasets | {pt.shape[1]} particles')
print('Asymptotes:', est_path)
print('Survival particles:', particle_path)
display(est[['n_cells_available', 'O_lambda_inf_central', 'O_lambda_inf_q025', 'O_lambda_inf_q975']].head(10))